# Fundamentos de *Deep Learning I* 
> Héctor J. Hortúa, PhD · Instituto de Neurociencias -SavIA-Lab
## Tensores y diferenciación automática con **JAX**

>  **Módulo 2 — Herramientas y manejo de datos**
> **Framework:** JAX

Este es el tercer notebook paralelo de la clase. **JAX** (de Google) adopta un enfoque distinto a TensorFlow y PyTorch: en lugar de un framework orientado a objetos con capas y estado mutable, ofrece **NumPy acelerado + transformaciones de funciones**. Aquí la diferenciación automática no es un detalle interno: es una **función que transforma funciones** (`jax.grad`), y ese es precisamente su mayor atractivo.

### Objetivos de aprendizaje

1. Entender la filosofía **funcional** de JAX y sus **arreglos inmutables**.
2. Usar `jax.numpy` como reemplazo casi directo de NumPy.
3. Manejar la **aleatoriedad explícita** con `PRNGKey` (muy distinto de NumPy/TF/Torch).
4. Aplicar las tres transformaciones clave: **`grad`** (derivar), **`jit`** (acelerar) y **`vmap`** (vectorizar).
5. Calcular gradientes, jacobianos y derivadas de orden superior con una elegancia notable.


## Filosofía: NumPy + transformaciones

JAX se resume en dos ideas:

1. **`jax.numpy` (`jnp`)** es una copia de la API de NumPy que corre en **CPU, GPU y TPU** sin cambiar el código.
2. Sobre cualquier función Python "pura" puedes aplicar **transformaciones componibles**:
   - **`grad(f)`** → devuelve una nueva función que calcula la derivada de `f`.
   - **`jit(f)`** → compila `f` con XLA para que vuele.
   - **`vmap(f)`** → vectoriza `f` sobre un eje de lote automáticamente.

La contrapartida es la **programación funcional**: las funciones deben ser **puras** (sin efectos secundarios) y los arreglos son **inmutables**. No hay `tf.Variable` ni `requires_grad`: el estado se pasa explícitamente como argumento. Esto hace el código más predecible y es ideal para investigación.

## Preparación del entorno

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap, jacobian
import numpy as np
import matplotlib.pyplot as plt

print("JAX:", jax.__version__)
print("Dispositivos:", jax.devices())

## Arreglos y rangos

Los arreglos de JAX (`jnp.ndarray`) se ven y operan como los de NumPy: mismo concepto de **rango** (número de ejes, `.ndim`), **forma** (`.shape`) y **tipo** (`.dtype`).

| Rango | Nombre | Ejemplo |
|---|---|---|
| 0 | Escalar | `4` |
| 1 | Vector | `[2, 3, 4]` |
| 2 | Matriz | tabla filas × columnas |
| 3+ | Tensor | imágenes, *batches* |

In [ ]:
escalar = jnp.array(4)
vector  = jnp.array([2.0, 3.0, 4.0])
matriz  = jnp.array([[1, 2], [3, 4], [5, 6]], dtype=jnp.float32)

print("escalar:", escalar, "| ndim:", escalar.ndim)
print("vector :", vector,  "| ndim:", vector.ndim)
print("matriz :\n", matriz, "\n| ndim:", matriz.ndim)

In [ ]:
rank_3 = jnp.array([
    [[0, 1, 2, 3, 4],   [5, 6, 7, 8, 9]],
    [[10,11,12,13,14],  [15,16,17,18,19]],
    [[20,21,22,23,24],  [25,26,27,28,29]],
])
print("shape:", rank_3.shape, "| ndim:", rank_3.ndim)
print(rank_3)

### Indexado y la clave: **inmutabilidad**

El indexado de lectura es igual que en NumPy. Pero **no puedes asignar en el sitio** (`x[0] = 5` da error): los arreglos de JAX son **inmutables**. En su lugar, se usa la sintaxis funcional **`x.at[indice].set(valor)`**, que devuelve un arreglo **nuevo**.

In [ ]:
print("Columna 4 de todas las capas:\n", rank_3[:, :, 4])

# Actualización inmutable: NO modifica rank_3, crea uno nuevo
nuevo = rank_3.at[0, 0, 0].set(999)
print("\nOriginal[0,0,0]:", rank_3[0, 0, 0], "(intacto)")
print("Nuevo[0,0,0]   :", nuevo[0, 0, 0], "(modificado en la copia)")

### Puente con NumPy

In [ ]:
arr = np.array(rank_3)          # JAX -> NumPy
print(type(arr), arr.shape)

de_numpy = jnp.array(np.array([[1.0, 2.0], [3.0, 4.0]]))  # NumPy -> JAX
print("Desde NumPy:\n", de_numpy)

## Operaciones sobre tensores

Elemento a elemento y matriciales, con la misma API de NumPy.

In [ ]:
a = jnp.array([[1, 2], [3, 4]])
b = jnp.ones((2, 2), dtype=jnp.int32)

print("Suma:\n", a + b)
print("Producto elemento a elemento:\n", a * b)
print("Producto MATRICIAL:\n", a @ b)

In [ ]:
c = jnp.array([[4.0, 5.0], [10.0, 1.0]])
print("máximo:", jnp.max(c))
print("media :", jnp.mean(c))
print("argmax:", jnp.argmax(c))

## Aleatoriedad explícita: `PRNGKey`

Aquí JAX se separa radicalmente de los demás. Para que las funciones sean **puras y reproducibles**, JAX **no tiene un estado global de aleatoriedad**. En su lugar, tú pasas una **llave** (`key`) explícita, y cada vez que necesitas números nuevos **divides** la llave con `split`. Puede parecer incómodo al principio, pero garantiza reproducibilidad perfecta.

In [ ]:
key = jax.random.PRNGKey(0)          # semilla explícita
key, subkey = jax.random.split(key)  # deriva una nueva llave
d = jax.random.uniform(subkey, shape=(200,), minval=-10.0, maxval=10.0)

plt.figure(figsize=(7, 3))
plt.plot(jax.nn.relu(d))
plt.title("ReLU aplicada a 200 valores aleatorios (JAX)")
plt.xlabel("índice"); plt.ylabel("relu(x)")
plt.show()

## `jnp.where`: condicionales vectorizados

Igual que en NumPy, `jnp.where` cubre dos usos:

- **`jnp.where(condicion)`** (o `jnp.argwhere`) devuelve las **posiciones** donde la condición es verdadera.
- **`jnp.where(condicion, a, b)`** elige elemento a elemento entre `a` y `b`: un *if* ternario sobre todo el arreglo.

En JAX esto es **especialmente importante**: dentro de funciones que vas a `jit` no puedes usar un `if` de Python sobre el *valor* de un arreglo. El flujo de control que depende de los datos se expresa con `jnp.where` (o `jax.lax.cond`).

In [ ]:
x = jnp.array([1, -2, 3, -4, 5])
print("posiciones donde x > 0:", jnp.argwhere(x > 0).flatten())   # [0 2 4]

### ReLU manual con `where`

In [ ]:
x = jnp.array([1, -2, 3, -4, 5])
print(jnp.where(x > 0, x, 0))   # [1 0 3 0 5]  -> reemplaza negativos por 0

### *If* ternario elemento a elemento

In [ ]:
x = jnp.array([[1, 2, 3],
               [4, 5, 6]])
print(jnp.where(x > 3, x, -x))   # si x>3 deja x, si no -x

## Propiedades, *reshape* y *broadcasting*

Todo idéntico a NumPy: `.shape`, `.ndim`, `.dtype`, `.size`; `jnp.reshape` con `-1`; y las mismas reglas de *broadcasting*.

In [ ]:
t = jnp.zeros((3, 2, 4, 5))
print("shape:", t.shape, "| ndim:", t.ndim, "| dtype:", t.dtype, "| size:", t.size)

print("\nReshape [3,2,5] -> [6,5]:\n", rank_3.reshape(6, 5))

x = jnp.array([1, 2, 3])
m = jnp.array([[1.0, 2.0], [3.0, 4.0]])
col = jnp.array([[1.0], [2.0]])
print("\nBroadcasting vector*escalar:", x * 3)
print("Broadcasting matriz + columna:\n", m + col)

## Diferenciación automática: `grad`, la estrella de JAX

En TensorFlow usábamos `GradientTape`; en PyTorch, `.backward()`. En JAX, derivar es aún más directo: **`grad` es una función que recibe una función y devuelve otra función** —su derivada—.

```python
df = grad(f)   # df(x) es f'(x)
```

Esto encaja con la mentalidad matemática: si $f:\mathbb{R}\to\mathbb{R}$, entonces $\nabla f$ es otra función. No hay estado, no hay cinta, no hay que marcar tensores. Solo funciones puras.

### Ejemplo mínimo: derivada de $y = x^2$

$\frac{dy}{dx} = 2x$, que en $x=3$ vale $6$.

In [ ]:
def f(x):
    return x**2

df = grad(f)             # df es la derivada de f
print("f(3)  =", f(3.0))
print("f'(3) =", df(3.0), "(esperado: 6.0)")

### Gradiente respecto de varios parámetros

En JAX, los parámetros de una red se agrupan en una estructura (por ejemplo, una tupla o diccionario) y `grad` deriva respecto del **primer argumento** por defecto. Reproducimos la capa lineal $y = xW + b$ con pérdida cuadrática media.

In [ ]:
def perdida(params, x):
    w, b = params
    y = x @ w + b
    return jnp.mean(y**2)

key = jax.random.PRNGKey(42)
w = jax.random.normal(key, (3, 2))
b = jnp.zeros(2)
x = jnp.array([[1.0, 2.0, 3.0]])

grad_fn = grad(perdida)              # deriva respecto de 'params'
dw, db = grad_fn((w, b), x)
print("Gradiente respecto de w (shape", dw.shape, "):\n", dw)
print("Gradiente respecto de b:", db)

### Visualizar una derivada: sigmoide y su pendiente

Aquí aparece la segunda transformación: **`vmap`**. `grad` deriva funciones **escalares**; para aplicarla a cada punto de un vector, la **vectorizamos** con `vmap` en lugar de escribir un bucle.

In [ ]:
def sigmoide(x):
    return 1.0 / (1.0 + jnp.exp(-x))

xs = jnp.linspace(-10, 10, 201)
ys = sigmoide(xs)
dys = vmap(grad(sigmoide))(xs)      # grad puntual, vectorizado con vmap

plt.figure(figsize=(7, 4))
plt.plot(xs, ys, label='σ(x)  (sigmoide)')
plt.plot(xs, dys, label="σ'(x)  (derivada)")
plt.legend(); plt.xlabel('x'); plt.grid(alpha=0.3)
plt.title("Una función y su derivada, con grad + vmap")
plt.show()

### Derivadas de orden superior: solo componer `grad`

Aquí JAX es imbatible en elegancia: la segunda derivada es `grad(grad(f))`, la tercera `grad(grad(grad(f)))`, etc. Para $f(x)=x^3$: $f'=3x^2$, $f''=6x$, $f'''=6$.

In [ ]:
def f(x):
    return x**3

print("f'(2)   =", grad(f)(2.0),            "(esperado 12)")
print("f''(2)  =", grad(grad(f))(2.0),      "(esperado 12)")
print("f'''(2) =", grad(grad(grad(f)))(2.0), "(esperado 6)")

### Sin "tape persistente": reutilizar `grad` es gratis

En TensorFlow hay que elegir entre una `GradientTape` normal (un solo uso) y una `persistent=True` (varios usos, liberando recursos a mano con `del tape`). En JAX ese dilema **no existe**: como `grad(f)` es solo una función pura, la llamas cuantas veces quieras y sobre las funciones que quieras, sin ningún estado que administrar.

Y para elegir **respecto de qué argumento** derivar —el análogo a marcar un parámetro como entrenable o congelado— se usa `argnums`.

In [ ]:
def f(x, y):
    return x**2 + x * y

df_dx = grad(f, argnums=0)     # derivar solo respecto de x
df_dy = grad(f, argnums=1)     # derivar solo respecto de y
print("df/dx en (3,4):", df_dx(3.0, 4.0), "(2x + y = 10)")
print("df/dy en (3,4):", df_dy(3.0, 4.0), "(x = 3)")
print("ambos a la vez:", grad(f, argnums=(0, 1))(3.0, 4.0))

### Jacobianos: derivadas de funciones vectoriales

Cuando la función devuelve un vector, su derivada es una **matriz jacobiana**. JAX la calcula directamente con `jacobian`. Aquí, $f(x,y) = [x^2y,\; 5x + \sin y]$.

In [ ]:
def f(v):
    x, y = v
    return jnp.array([x**2 * y, 5 * x + jnp.sin(y)])

J = jacobian(f)(jnp.array([1.0, 2.0]))
print("Matriz jacobiana en (1, 2):\n", J)

## Verificación de una solución analítica de una EDO con `grad`

Comprobamos si $y(t)=\cos(t)$ resuelve la EDO $y''+y=0$. Aquí JAX brilla: la segunda derivada es literalmente `grad(grad(y))`, y la aplicamos a todo el vector de tiempos con `vmap`. Evaluamos el **residuo** $y''+y$, que debe ser $\approx 0$. Esta idea es la base de las *Physics-Informed Neural Networks* (PINNs).

In [ ]:
def y_fn(t):
    return jnp.cos(t)

t = jnp.linspace(0.0, 2 * jnp.pi, 200)

y   = vmap(y_fn)(t)
dy  = vmap(grad(y_fn))(t)           # primera derivada
d2y = vmap(grad(grad(y_fn)))(t)     # segunda derivada: ¡solo componer grad!
residuo = d2y + y                   # debería ser ~0

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(t, y,   'b-', label=r'$y=\cos t$')
axs[0].plot(t, dy,  'g-', label=r"$y'$")
axs[0].plot(t, d2y, 'm-', label=r"$y''$")
axs[0].legend(); axs[0].grid(alpha=0.3); axs[0].set_title("Función y sus derivadas")
axs[1].plot(t, residuo, 'r-')
axs[1].set_title(r"Residuo $y''+y$ (debe ser ~0)"); axs[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Residuo máximo:", jnp.max(jnp.abs(residuo)))

## `jit`: compilación que acelera

La tercera transformación, **`jit`**, compila una función con XLA. La primera llamada traza y compila (lenta); las siguientes vuelan. Se combina perfectamente con `grad`: en un entrenamiento real usarás `jit(grad(perdida))` para que cada paso sea rapidísimo.

In [ ]:
def costosa(x):
    return jnp.sum(x**2 + jnp.sin(x))

rapida = jit(costosa)               # versión compilada
v = jnp.arange(1000.0)

import timeit
# La primera llamada compila; forzamos la compilación antes de medir
_ = rapida(v).block_until_ready()
print("Sin jit:", timeit.timeit(lambda: costosa(v).block_until_ready(), number=1000))
print("Con jit:", timeit.timeit(lambda: rapida(v).block_until_ready(), number=1000))

> **La combinación mágica de JAX:** `jit(grad(vmap(f)))`. Compilas la versión vectorizada del gradiente de tu función. Esa composición de transformaciones es exactamente lo que hacen frameworks de redes construidos sobre JAX, como **Flax** y **Haiku**, que verás si profundizas en investigación.

## MCMC: Metropolis-Hastings desde cero

Cerramos con un muestreador **Metropolis-Hastings** de una Normal estándar $\mathcal{N}(0,1)$. En JAX luce distinto por la **aleatoriedad explícita**: cada paso consume una llave `key` que dividimos con `split`, lo que hace la cadena **perfectamente reproducible**.

1. Estado inicial $x_0$.
2. Propuesta $x' = x + \epsilon$, con $\epsilon \sim \mathcal{N}(0,\sigma^2)$.
3. Aceptar con probabilidad $\alpha=\min(1, p(x')/p(x))$; en log: $\log u < \log\alpha$.

> Usamos un bucle de Python por claridad. La versión idiomática y rápida en JAX emplearía `jax.lax.scan`, que compila toda la cadena de una vez.

In [ ]:
def log_prob(x):                      # log-densidad Normal(0,1) sin normalizar
    return -0.5 * x**2

def mh_step(x, key, sigma=0.5):
    k_prop, k_acc = jax.random.split(key)
    x_prop = x + sigma * jax.random.normal(k_prop)
    log_alpha = log_prob(x_prop) - log_prob(x)
    log_u = jnp.log(jax.random.uniform(k_acc))
    aceptar = log_u < log_alpha
    return jnp.where(aceptar, x_prop, x), aceptar

def sample_chain(n=10000, burnin=5000, x0=5.0, sigma=0.5, seed=0):
    key = jax.random.PRNGKey(seed)
    x = jnp.float32(x0)
    muestras, aceptados = [], 0
    for i in range(n + burnin):
        key, sub = jax.random.split(key)
        x, acc = mh_step(x, sub, sigma)
        if i >= burnin:
            muestras.append(float(x)); aceptados += int(acc)
    return np.array(muestras), aceptados / n

samples, tasa = sample_chain()
print(f"Tasa de aceptación: {tasa:.3f}  (ideal ~0.23-0.45)")
print(f"Media: {samples.mean():.4f} (esp. 0) | Desv: {samples.std():.4f} (esp. 1)")

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].plot(samples, linewidth=0.5); axs[0].set_title("Traceplot")
axs[0].set_xlabel("iteración"); axs[0].set_ylabel("x")
axs[1].hist(samples, bins=50, density=True, alpha=0.6, label="MCMC")
xg = np.linspace(-4, 4, 200)
axs[1].plot(xg, np.exp(-0.5 * xg**2) / np.sqrt(2 * np.pi), 'r-', label="Normal(0,1)")
axs[1].set_title("Distribución muestreada"); axs[1].legend()
plt.tight_layout(); plt.show()

## Resumen y comparación de los tres frameworks

| Concepto | TensorFlow | PyTorch | JAX |
|---|---|---|---|
| Arreglo base | `tf.Tensor` | `torch.Tensor` | `jnp.ndarray` |
| Mutabilidad | inmutable / `tf.Variable` | mutable | **inmutable** (`.at[].set()`) |
| Estado entrenable | `tf.Variable` | `requires_grad=True` | se pasa como argumento |
| Autodiff | `tf.GradientTape` | `.backward()` + `.grad` | `grad(f)` |
| Derivada de orden n | anidar cintas | `create_graph=True` | `grad(grad(...))` |
| Acelerar | `@tf.function` | `torch.compile` | `jit` |
| Vectorizar | `tf.vectorized_map` | `torch.vmap` | `vmap` |
| Estilo | OO / imperativo | OO / imperativo | **funcional** |

**Idea final:** los tres calculan lo mismo. JAX lo expresa como transformaciones de funciones puras, lo que resulta elegante para investigación; TensorFlow y PyTorch ofrecen APIs de alto nivel (Keras, `torch.nn`) más cómodas para construir y desplegar redes.